# **Nemzetközi MI Diákolimpia Online Forduló (2025) - Starter Notebook**

Ez a **Colab Notebook** a *2025-ös Nemzetközi MI Diákolimpia* online fordulójához készült, és egy alapvető kiindulópontot biztosít a versenyzők számára.  

A notebook tartalmazza az adatok betöltését és alapvető vizualizációját, valamint egy egyszerű **logisztikus regresszió** alapú baseline modellt. Ez a modell az **"Iskola Presztízse"** változó alapján próbálja megjósolni, hogy egy diákot felvesznek-e egyetemre vagy sem.  

A verseny során bármilyen csomagot vagy keretrendszert használhattok, amennyiben a feltöltött megoldás megfelel a **Kaggle versenyoldalon** megfogalmazott szabályoknak.  

A versenyhez biztosított adatok és a feladat is **teljesen szintetikus**, így nincs értelme külső forrásból adatokat keresni.  


# **0. Szükséges Csomagok Betöltése**

In [ ]:
import folium
import json
import torch
import gdown
import numpy as np
import geopandas as gpd
import pandas as pd
from sklearn.linear_model import LogisticRegression

device = "cuda" if torch.cuda.is_available() else "cpu"

# **1. Adatok Betöltése**

A verseny során három fájl áll rendelkezésetekre:

- `train.csv` - A tanításhoz szükséges adatokat tartalmazza, beleértve minden jelentkezőhöz tartozó jellemzőt és a célváltozót (`Felvételi Eredmény`), amely jelzi, hogy az adott diák felvételt nyert-e az egyetemre (1 - felvették, 0 - nem vették fel). Minden sor egyedi `ID`-val azonosított.

- `test.csv` - A verseny értékeléséhez használt tesztadatokat tartalmazza, a célváltozó (`Felvételi Eredmény`) **nélkül**. A célotok, hogy erre a halmazra előrejelzést készítsetek. A predikció során minden sorhoz tartozik egy `ID`, amely alapján visszaköthető a megfigyeléshez.

- `counties.geojson` - Egy GeoJSON formátumú fájl, amely Magyarország megyéinek **földrajzi határait és középponti koordinátáit** tartalmazza. Ez lehetőséget ad arra, hogy a jelentkezők lakhelyét (megye szinten) térinformatikai adatokkal egészítsétek ki, például kiszámolva a lakóhely és az egyetem közötti távolságot.

---

A feladatotok, hogy a `train.csv` adathalmaz és a `counties.geojson` térképi adatai alapján **megtanítsatok egy gépi tanulási modellt**, amely képes megbecsülni, hogy a `test.csv` fájlban szereplő diákokat felvennék-e az egyetemre.

A modellnek minden `test.csv`-beli `ID`-hez egy **0 és 1 közötti valószínűségi értéket** kell visszaadnia. Ez az érték a rendszer szerint annak valószínűségét fejezi ki, hogy az adott diák felvételt nyer (1 - biztos felvétel, 0 - biztos elutasítás).

## **1.1. Földrajzi Adatok**

A feladat megoldásához egy `.geojson` fájl is rendelkezésre áll, amely tartalmazza Magyarország megyéinek **határait és földrajzi koordinátáit** (+ Budapest, aminek jogállása főváros és nem megye) a **WGS84 (World Geodetic System 1984) koordináta-rendszerben**. A fájl az egyes megyéket **földrajzi szélesség (latitude) és földrajzi hosszúság (longitude)** értékekkel írja le. [GeoJSON](https://geojson.org/)

In [ ]:
with open('/kaggle/input/magyar-mi-diakolimpia-online-valogato-2025/counties.geojson', 'r', encoding='utf-8') as f:
    geojson_data = json.load(f)

print(f"Vármegyék Száma + Budapest: {len(geojson_data['features'])}")

## **1.2 Budapest főváros struktúrájának megtekintése**

In [ ]:
budapest_data = geojson_data['features'][0]
print(budapest_data.keys())

print(f'Budapesthez tartozó adatok: {budapest_data}')
print(f'Fővárost leíró poligon: {budapest_data["geometry"]}')
print(f'Fővárost leíró tulajdonságok: {budapest_data["properties"]}')

In [ ]:
print(f'Fővárost leíró poligon első koordinátája: {budapest_data["geometry"]["coordinates"][0][0]}')
print(f'Fővárost leíró poligon második koordinátája: {budapest_data["geometry"]["coordinates"][0][1]}')
print(f'Főváros neve: {budapest_data["properties"]["megye"]}')

## **1.3 A GeoJSON file megjelenítése *folium* segítségével**

In [ ]:
gdf = gpd.GeoDataFrame.from_features(geojson_data["features"])

m = folium.Map(location=[47.0, 19.5], zoom_start=7)
for _, row in gdf.iterrows():
    folium.GeoJson(row["geometry"]).add_to(m)

uni_coords = (47.472113, 19.062236)

folium.Marker(
    location=uni_coords,
    popup="University (Budapest)",
    icon=folium.Icon(color="red", icon="graduation-cap", prefix="fa"),
).add_to(m)

m

## **1.4 Egyetemi Távolság**

Az egyik térképész barátod — aki egyébként korábban az egyetem adminisztrációs rendszerén dolgozott — egy kávé mellett elárulta neked, hogy a **felvételi döntések hátterében** bizony **figyelembe veszik a jelentkezők lakóhelyének távolságát is az egyetemtől**.

Bár hivatalosan erről nem esik szó, az informális beszélgetés során megtudtad, hogy **a túl távolról érkezők kisebb eséllyel jutnak be**, részben logisztikai okokból, részben a hallgatói lemorzsolódás kockázata miatt.

A térképész haverod ráadásul nem hagyott cserben: küldött neked egy **függvényt**, amellyel a jelentkezők megyeszintű lakhelyei alapján **pontosan kiszámolhatod a távolságot az egyetemhez**. Így lehetőséged nyílik beépíteni ezt az információt a saját predikciós modelledbe is, pont úgy, ahogyan azt a bizottság is megteszi.


In [ ]:
def haversine(lat1: float, lon1: float, lat2:float, lon2:float) -> float:
    """
    Kiszámítja a Haversine-formula segítségével a két földrajzi pont közötti távolságot kilométerben.

    A Haversine-formula egy gömbi trigonometrián alapuló számítás, amely két földrajzi koordináta
    (szélesség és hosszúság) közötti nagy köríves távolságot adja meg.

    Paraméterek:
        lat1 (float): Az első pont földrajzi szélessége (fokban).
        lon1 (float): Az első pont földrajzi hosszúsága (fokban).
        lat2 (float): A második pont földrajzi szélessége (fokban).
        lon2 (float): A második pont földrajzi hosszúsága (fokban).

    Visszatérési érték:
        float: A két pont közötti távolság kilométerben.

    Példa:
        >>> haversine(47.4979, 19.0402, 48.8566, 2.3522)
        1244.51  # Budapest és Párizs közötti távolság kilométerben (megközelítőleg)
    """
    R = 6371  # A Föld sugara kilométerben
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2.0)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2.0)**2
    return 2 * R * np.arcsin(np.sqrt(a))

## **1.5 Tanuló és Teszt Adatok**

In [ ]:
train_df = pd.read_csv('/kaggle/input/magyar-mi-diakolimpia-online-valogato-2025/train.csv')
test_df = pd.read_csv('/kaggle/input/magyar-mi-diakolimpia-online-valogato-2025/test.csv')

## **1.6 Oszlopok Leírásai**

- `Életkor` *(int)* – A jelentkező életkora években. Lehetséges értékek: 17, 18 vagy 19.
- `Nem` *(int)* – A jelentkező neme: 0 - férfi, 1 - nő.
- `Osztályzat_9` *(float)* – A 9. osztályos tanulmányi átlag 2.00 és 5.00 között, 0.25-ös lépésközzel.
- `Osztályzat_10` *(float)* – A 10. osztályos tanulmányi átlag 2.00 és 5.00 között, 0.25-ös lépésközzel.
- `Osztályzat_11` *(float)* – A 11. osztályos tanulmányi átlag 2.00 és 5.00 között, 0.25-ös lépésközzel.
- `Osztályzat_12` *(float)* – A 12. osztályos tanulmányi átlag 2.00 és 5.00 között, 0.25-ös lépésközzel.
- `Történelem` *(int)* – Történelem érettségi pontszám (0–100).
- `Matematika` *(int)* – Matematika érettségi pontszám (0–100).
- `Magyar Nyelv és Irodalom` *(int)* – Magyar nyelv és irodalom érettségi pontszám (0–100).
- `Informatika` *(int)* – Informatika érettségi pontszám (0–100), ha választotta; különben -1.
- `Biológia` *(int)* – Biológia érettségi pontszám (0–100), ha választotta; különben -1.
- `Fizika` *(int)* – Fizika érettségi pontszám (0–100), ha választotta; különben -1.
- `Angol` *(int)* – Angol érettségi pontszám (0–100), ha választotta; különben -1.
- `Német` *(int)* – Német érettségi pontszám (0–100), ha választotta; különben -1.
- `Informatika_emelt` *(int)* – 1, ha az informatika tárgyból emelt szintű érettségit tett; 0, ha középszintűt; -1, ha nem választotta a tárgyat.
- `Biológia_emelt` *(int)* – 1, ha biológiából emelt szintű érettségit tett; 0, ha középszintűt; -1, ha nem választotta.
- `Fizika_emelt` *(int)* – 1, ha fizikából emelt szintű érettségit tett; 0, ha középszintűt; -1, ha nem választotta.
- `Angol_emelt` *(int)* – 1, ha angolból emelt szintű érettségit tett; 0, ha középszintűt; -1, ha nem választotta.
- `Német_emelt` *(int)* – 1, ha németből emelt szintű érettségit tett; 0, ha középszintűt; -1, ha nem választotta.
- `Matematika_emelt` *(int)* – 1, ha a matematikából kapott pontszámot emelt szinten szerezte; 0, ha középszinten.
- `Történelem_emelt` *(int)* – 1, ha történelemből emelt szintű érettségit tett; 0, ha középszintűt.
- `Magyar Nyelv és Irodalom_emelt` *(int)* – 1, ha magyarból emelt szintű érettségit tett; 0, ha középszintűt.
- `Szülői Végzettség` *(int)* – A szülők legmagasabb iskolai végzettsége: 1 - legfeljebb 8 osztály, 2 - szakmunkás, 3 - érettségi, 4 - főiskola, 5 - egyetem.
- `Középiskola Presztízse` *(float)* – A diák középiskolájának presztízsértéke 1 és 10 között, ahol 10 a legtekintélyesebb.
- `Extrakurrikuláris Tevékenységek` *(int)* – Részt vett-e szakkörön, sporton, egyéb tevékenységen: 1 - igen, 0 - nem.
- `Tanulási Szokások` *(int)* – A tanulási szokások minősége 1-től 8-ig: 1–3 rendszertelen, 4–5 átlagos, 6–8 következetes, tudatos tanulás.
- `Munkatapasztalat` *(int)* – Van-e korábbi munkatapasztalata: 1 - igen, 0 - nem.
- `Ajánlások Száma` *(int)* – A diák által megszerzett ajánlások száma (1–6), pl. tanári vagy szakmai mentor ajánlások.
- `Versenyeken Való Részvétel` *(int)* – Részt vett-e tanulmányi versenyeken: 1 - igen, 0 - nem.
- `Vármegye` *(string)* – A jelentkező lakhelye szöveges formában (pl. "Baranya", "Pest").
- `Stressz Szint` *(float)* – A jelentkező szubjektív stressz-szintje 0.0 és 1.0 között, ahol 1.0 a maximális stressz.
- `Felvételi Eredmény` *(int)* – A **bináris célváltozó**: 1, ha a jelentkezőt felvették az egyetemre; 0, ha nem.

In [ ]:
admitted_count = train_df['Felvételi Eredmény'].sum()
print(f"Felvettek Száma a Tanító Adathalmazból: {admitted_count}")
print(f'Tanitó Adathalmaz Elemeinek Száma: {len(train_df)}')

In [ ]:
train_df.head()

# **2. Példa Megoldás**

## **2.1 Logisztikus Regresszió**

In [ ]:
X_train = train_df[['Középiskola Presztízse']]
y_train = train_df['Felvételi Eredmény']

model = LogisticRegression()
model.fit(X_train, y_train)

In [ ]:
X_test = test_df[['Középiskola Presztízse']]

y_pred_test = model.predict(X_test)
y_pred_proba_test = model.predict_proba(X_test)[:, 1]

## **2.2 Predikciók Kimentése Megfelelő Formátumban**

In [ ]:
test_idx = test_df['ID']
test_output_df = pd.DataFrame({'ID': test_idx, 'Felvételi Eredmény': y_pred_proba_test})
test_output_df.to_csv('HUN123_1.csv', index=False)

In [ ]:
test_output_df.head()

# **Predikciók Feltöltése - Beküldési Útmutató**

Amennyiben úgy döntötök, hogy a predikciókat tartalmazó `.csv` fájlt feltöltitek, kérjük, figyeljetek az alábbi formátumokra és elnevezési szabályokra:

### **Példa CSV Formátum**

```csv
ID,Felvételi Eredmény
20079,0.405314
20080,0.401229
20081,0.401746
20082,0.402005
20083,0.397336
...
```

### **Fájlnévadási konvenció**

- A fájl neve legyen a következő formátumban: *\<kapott_azonosító>_\<checkpoint_szám>.csv*:
  - `<kapott_azonosító>` a kapott egyedi azonosító, amit az online verseny előtt 20 perccel kiküldünk mindenkinek (pl. `HUN123`)
  - `<checkpoint_szám>` az adott próbálkozás vagy feltöltés sorszáma (pl. `1`, `2`, stb.)

**Példa**: `HUN123_1.csv`

- A predikcióhoz tartozó notebook fájlt is **ugyanezen konvenció alapján** nevezzétek el és mentsétek `.ipynb` formátumban:

**Példa**: `HUN123_1.ipynb`

### **Fájlmentés**

- A `.ipynb` notebook fájlt mentsétek le **lokálisan a gépetekre**.
- Ez azért szükséges, mert a kiválasztott feltöltéshez tartozó notebookot később is be kell tudnotok küldeni (lásd következő pont).

### **Beküldés véglegesítése**

- **A Kaggle-re való regisztráció során azt az e-mail címet használjátok, amelyet a versenyre való regisztrációnál is megadtatok (Gmail használata ajánlott).**  
- A **Google Form kitöltéséhez is ugyanezt az e-mail címet kell használnotok**, hogy a feltöltések egyértelműen beazonosíthatóak legyenek. (Amennyiben a Google-fiók használata nem megoldható, a végleges megoldásokat a **mi_olimpia@inf.elte.hu** e-mail címre is beküldhetitek.  
Kérjük, hogy az e-mailben adjátok meg a teljes neveteket és a Kaggle felhasználóneveteket, hogy a feltöltések egyértelműen beazonosíthatóak legyenek.)
- **A Kaggle platformon a felhasználónevetek a saját nevetek legyen** a verseny idejére, mivel **álnevekkel és alias nevekkel történő feltöltéseket nem tudunk elfogadni**.
- A verseny **lezárása után 10 percen belül** be kell küldenetek a kiválasztott feltöltéshez tartozó notebookot is egy **Google Űrlapon keresztül**. Az űrlapot csak egyszer töltsétek ki!
- A kiválasztott feltöltés és beadott notebook alapján történik az ellenőrzés és a privát pontszám kiszámítása.

### **Több feltöltés**

- Több próbálkozás esetén használjatok növekvő checkpoint számot (pl. `HUN123_2.csv`, `HUN123_3.csv` stb.)
- Minden `.csv` fájlhoz tartozzon külön mentett `.ipynb` fájl is, azonos névvel.

---

**Ne felejtsétek el:** a notebook és a predikciós fájl *páronként* érvényes, így mindig ügyeljetek a pontos fájlelnevezésre és mentésre!

**Google Forms:** [Link](https://forms.gle/Fsjjpiky1s72C7ta8)
